# 피처 셀렉션 결과 괜찮은지 확인용도 (smote 비율은 1로만 두고 실행함)

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np

from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH = "../10,11,12번 [Train].parquet"
TEST_PATH  = "../10,11,12번 [Test].parquet"

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.5


# ============================================================
# 2. 피처 파일
# ============================================================

feature_files = {
    "VIF5":           "../13번. 최종피처(VIF 5미만).parquet",
    "VIF10":          "../13번. 최종피처(VIF 10미만).parquet",
    "VIF10_ALPHA005": "../13번. 최종피처(VIF 10미만,alpha=0.05).parquet",
    "VIF5_AIC":       "../13번. 최종피처(VIF5미만+AIC).parquet"
}


# ============================================================
# 3. 모델 정의
# ============================================================

models = {

    "LogisticRegression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs",
        max_iter=1000, random_state=RANDOM_STATE
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbose=-1
    ),

    "SVM": SVC(
        kernel="rbf", C=1.0, gamma="scale",
        probability=True, class_weight="balanced",
        random_state=RANDOM_STATE
    )
}


# ============================================================
# 4. 데이터 로드
# ============================================================

train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

y_train = train[TARGET_COL]
y_test  = test[TARGET_COL]


# ============================================================
# 5. 모든 변수 합집합 생성
# ============================================================

all_features = set()

for feature_name, feature_path in feature_files.items():
    feature_df  = pd.read_parquet(feature_path)
    feature_col = (
        "final_feature" if "final_feature" in feature_df.columns else "feature"
    )
    all_features.update(feature_df[feature_col].tolist())

all_features = [f for f in all_features if f in train.columns]

print("="*70)
print(f"전체 변수 개수 : {len(all_features)}")
print("="*70)


# ============================================================
# 6. 오버샘플링 함수 정의
# ============================================================

def apply_none(X_train, y_train, ratio=None):
    """클래스 불균형 처리 없음 - 원본 그대로 반환"""
    df_res = X_train.copy()
    df_res[TARGET_COL] = y_train.values
    return df_res


def apply_borderline_smote(X_train, y_train, ratio):
    """BorderlineSMOTE 적용"""
    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_res, y_res = smote.fit_resample(X_train, y_train)
    df_res = pd.DataFrame(X_res, columns=X_train.columns)
    df_res[TARGET_COL] = y_res
    return df_res


def apply_ctgan(X_train, y_train, ratio):
    """CTGAN 적용"""
    n_majority = (y_train == 0).sum()
    n_minority = (y_train == 1).sum()
    n_target   = int(n_majority * ratio)
    n_to_gen   = max(n_target - n_minority, 0)

    if n_to_gen == 0:
        df_res = X_train.copy()
        df_res[TARGET_COL] = y_train.values
        return df_res

    minority_df = X_train[y_train == 1].copy()
    minority_df[TARGET_COL] = 1

    ctgan = CTGAN(epochs=100, verbose=False)
    ctgan.fit(minority_df, discrete_columns=[TARGET_COL])

    synthetic = ctgan.sample(n_to_gen)
    synthetic[TARGET_COL] = 1

    original_df = X_train.copy()
    original_df[TARGET_COL] = y_train.values

    df_res = pd.concat([original_df, synthetic], ignore_index=True)
    return df_res


# ============================================================
# 7. 실험 루프
#    방식별 ratio 설정이 다르므로 method_configs로 관리
# ============================================================

method_configs = {
    "None"            : (apply_none,              [None]),
    "BorderlineSMOTE" : (apply_borderline_smote,  [1.0]),   # 0.2~1.0 → 1.0만
    "CTGAN"           : (apply_ctgan,             [1.0])    # 0.2~1.0 → 1.0만
}

results = []

for method_name, (oversample_fn, ratios) in method_configs.items():

    print("\n")
    print("="*70)
    print(f"오버샘플링 방식 : {method_name}")
    print("="*70)

    for ratio in ratios:

        print(f"\n  ratio = {ratio}")

        X_full          = train[all_features]
        train_resampled = oversample_fn(X_full, y_train, ratio)

        n0 = (train_resampled[TARGET_COL] == 0).sum()
        n1 = (train_resampled[TARGET_COL] == 1).sum()
        print(f"  데이터 구성 → 0: {n0}, 1: {n1}")

        for feature_name, feature_path in feature_files.items():

            feature_df  = pd.read_parquet(feature_path)
            feature_col = (
                "final_feature" if "final_feature" in feature_df.columns
                else "feature"
            )
            use_features = [
                f for f in feature_df[feature_col].tolist()
                if f in train_resampled.columns
            ]

            X_train_final = train_resampled[use_features]
            y_train_final = train_resampled[TARGET_COL]
            X_test_final  = test[use_features]

            for model_name, model in models.items():

                print(f"    [{feature_name}] {model_name} 학습 중...")

                model.fit(X_train_final, y_train_final)

                y_prob = model.predict_proba(X_test_final)[:, 1]
                y_pred = (y_prob >= THRESHOLD).astype(int)

                results.append({
                    "Method"      : method_name,
                    "SMOTE_Ratio" : ratio if ratio is not None else "-",
                    "Feature_Set" : feature_name,
                    "Model"       : model_name,
                    "Num_Features": len(use_features),
                    "Accuracy"    : accuracy_score(y_test, y_pred),
                    "ROC_AUC"     : roc_auc_score(y_test, y_prob),
                    "PR_AUC"      : average_precision_score(y_test, y_prob),
                    "F1"          : f1_score(y_test, y_pred),
                    "Precision"   : precision_score(y_test, y_pred),
                    "Recall"      : recall_score(y_test, y_pred)
                })


# ============================================================
# 8. 결과 출력
# ============================================================

result_df = pd.DataFrame(results).round(4)

print("\n")
print("="*100)
print("전체 실험 결과")
print("="*100)
print(result_df.to_string(index=False))

# 핵심 지표별 상위 5개
for metric in ["PR_AUC", "Recall", "F1", "ROC_AUC"]:
    print(f"\n[{metric} 상위 5개]")
    print(
        result_df[["Method", "SMOTE_Ratio", "Feature_Set", "Model", metric]]
        .sort_values(metric, ascending=False)
        .head(5)
        .to_string(index=False)
    )

# 핵심 비교 : 방식별 평균 성능
print("\n")
print("="*70)
print("오버샘플링 방식별 평균 성능 비교")
print("="*70)
print(
    result_df.groupby("Method")[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)

# 방식 × 모델별 평균 성능
print("\n")
print("="*70)
print("방식 × 모델별 평균 성능")
print("="*70)
print(
    result_df.groupby(["Method", "Model"])[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)


# ============================================================
# 9. 저장
# ============================================================

result_df.to_csv(
    "Oversample_Method_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료 : Oversample_Method_Comparison.csv")

# 피처셀렉션 4가지 결과에 따라서,  클래스 불균형 방식에 따라 달라지는 성능평가 결과 비교

## 머신러닝 - Logistic, RandomForest, XGB, LGBM, SVM

In [ ]:
# ============================================================
# 0. 라이브러리
# ============================================================

import pandas as pd
import numpy as np

from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.svm import SVC

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

import warnings
warnings.filterwarnings("ignore")


# ============================================================
# 1. 설정값
# ============================================================

TRAIN_PATH = "../10,11,12번 [Train].parquet"
TEST_PATH  = "../10,11,12번 [Test].parquet"

TARGET_COL   = "부실라벨_ICR3년"
RANDOM_STATE = 42
THRESHOLD    = 0.5


# ============================================================
# 2. 피처 파일
# ============================================================

feature_files = {
    "VIF5":           "../13번. 최종피처(VIF 5미만).parquet",
    "VIF10":          "../13번. 최종피처(VIF 10미만).parquet",
    "VIF10_ALPHA005": "../13번. 최종피처(VIF 10미만,alpha=0.05).parquet",
    "VIF5_AIC":       "../13번. 최종피처(VIF5미만+AIC).parquet"
}


# ============================================================
# 3. 모델 정의
# ============================================================

models = {

    "LogisticRegression": LogisticRegression(
        penalty="l2", C=1.0, solver="lbfgs",
        max_iter=1000, random_state=RANDOM_STATE
    ),

    "RandomForest": RandomForestClassifier(
        n_estimators=300, max_depth=4, min_samples_leaf=5,
        class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        use_label_encoder=False, eval_metric="aucpr",
        random_state=RANDOM_STATE, verbosity=0
    ),

    "LightGBM": LGBMClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=RANDOM_STATE, verbose=-1
    ),

    "SVM": SVC(
        kernel="rbf", C=1.0, gamma="scale",
        probability=True, class_weight="balanced",
        random_state=RANDOM_STATE
    )
}


# ============================================================
# 4. 데이터 로드
# ============================================================

train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

y_train = train[TARGET_COL]
y_test  = test[TARGET_COL]


# ============================================================
# 5. 모든 변수 합집합 생성
# ============================================================

all_features = set()

for feature_name, feature_path in feature_files.items():
    feature_df  = pd.read_parquet(feature_path)
    feature_col = (
        "final_feature" if "final_feature" in feature_df.columns else "feature"
    )
    all_features.update(feature_df[feature_col].tolist())

all_features = [f for f in all_features if f in train.columns]

print("="*70)
print(f"전체 변수 개수 : {len(all_features)}")
print("="*70)


# ============================================================
# 6. 오버샘플링 함수 정의
# ============================================================

def apply_none(X_train, y_train, ratio=None):
    """클래스 불균형 처리 없음 - 원본 그대로 반환"""
    df_res = X_train.copy()
    df_res[TARGET_COL] = y_train.values
    return df_res


def apply_borderline_smote(X_train, y_train, ratio):
    """BorderlineSMOTE 적용"""
    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_res, y_res = smote.fit_resample(X_train, y_train)
    df_res = pd.DataFrame(X_res, columns=X_train.columns)
    df_res[TARGET_COL] = y_res
    return df_res


def apply_ctgan(X_train, y_train, ratio):
    """CTGAN 적용"""
    n_majority = (y_train == 0).sum()
    n_minority = (y_train == 1).sum()
    n_target   = int(n_majority * ratio)
    n_to_gen   = max(n_target - n_minority, 0)

    if n_to_gen == 0:
        df_res = X_train.copy()
        df_res[TARGET_COL] = y_train.values
        return df_res

    minority_df = X_train[y_train == 1].copy()
    minority_df[TARGET_COL] = 1

    ctgan = CTGAN(epochs=100, verbose=False)
    ctgan.fit(minority_df, discrete_columns=[TARGET_COL])

    synthetic = ctgan.sample(n_to_gen)
    synthetic[TARGET_COL] = 1

    original_df = X_train.copy()
    original_df[TARGET_COL] = y_train.values

    df_res = pd.concat([original_df, synthetic], ignore_index=True)
    return df_res


# ============================================================
# 7. 실험 루프
#    방식별 ratio 설정이 다르므로 method_configs로 관리
# ============================================================

method_configs = {
    # 방식명       : (함수,                   ratio 리스트)
    "None"         : (apply_none,             [None]),          # ratio 의미 없음
    "BorderlineSMOTE": (apply_borderline_smote, [0.2, 0.3, 0.5, 0.7, 1.0]),
    "CTGAN"        : (apply_ctgan,            [0.2, 0.3, 0.5, 0.7, 1.0])
}

results = []

for method_name, (oversample_fn, ratios) in method_configs.items():

    print("\n")
    print("="*70)
    print(f"오버샘플링 방식 : {method_name}")
    print("="*70)

    for ratio in ratios:

        print(f"\n  ratio = {ratio}")

        X_full          = train[all_features]
        train_resampled = oversample_fn(X_full, y_train, ratio)

        n0 = (train_resampled[TARGET_COL] == 0).sum()
        n1 = (train_resampled[TARGET_COL] == 1).sum()
        print(f"  데이터 구성 → 0: {n0}, 1: {n1}")

        for feature_name, feature_path in feature_files.items():

            feature_df  = pd.read_parquet(feature_path)
            feature_col = (
                "final_feature" if "final_feature" in feature_df.columns
                else "feature"
            )
            use_features = [
                f for f in feature_df[feature_col].tolist()
                if f in train_resampled.columns
            ]

            X_train_final = train_resampled[use_features]
            y_train_final = train_resampled[TARGET_COL]
            X_test_final  = test[use_features]

            for model_name, model in models.items():

                print(f"    [{feature_name}] {model_name} 학습 중...")

                model.fit(X_train_final, y_train_final)

                y_prob = model.predict_proba(X_test_final)[:, 1]
                y_pred = (y_prob >= THRESHOLD).astype(int)

                results.append({
                    "Method"      : method_name,
                    "SMOTE_Ratio" : ratio if ratio is not None else "-",
                    "Feature_Set" : feature_name,
                    "Model"       : model_name,
                    "Num_Features": len(use_features),
                    "Accuracy"    : accuracy_score(y_test, y_pred),
                    "ROC_AUC"     : roc_auc_score(y_test, y_prob),
                    "PR_AUC"      : average_precision_score(y_test, y_prob),
                    "F1"          : f1_score(y_test, y_pred),
                    "Precision"   : precision_score(y_test, y_pred),
                    "Recall"      : recall_score(y_test, y_pred)
                })


# ============================================================
# 8. 결과 출력
# ============================================================

result_df = pd.DataFrame(results).round(4)

print("\n")
print("="*100)
print("전체 실험 결과")
print("="*100)
print(result_df.to_string(index=False))

# 핵심 지표별 상위 5개
for metric in ["PR_AUC", "Recall", "F1", "ROC_AUC"]:
    print(f"\n[{metric} 상위 5개]")
    print(
        result_df[["Method", "SMOTE_Ratio", "Feature_Set", "Model", metric]]
        .sort_values(metric, ascending=False)
        .head(5)
        .to_string(index=False)
    )

# 핵심 비교 : 방식별 평균 성능
print("\n")
print("="*70)
print("오버샘플링 방식별 평균 성능 비교")
print("="*70)
print(
    result_df.groupby("Method")[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)

# 방식 × 모델별 평균 성능
print("\n")
print("="*70)
print("방식 × 모델별 평균 성능")
print("="*70)
print(
    result_df.groupby(["Method", "Model"])[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)


# ============================================================
# 9. 저장
# ============================================================

result_df.to_csv(
    "Oversample_Method_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료 : Oversample_Method_Comparison.csv")

전체 변수 개수 : 71


SMOTE sampling_strategy = 0.2
  After SMOTE → 0: 31087, 1: 6217
  [VIF5] LogisticRegression 학습 중...
  [VIF5] XGBoost 학습 중...
  [VIF5] LightGBM 학습 중...
  [VIF10] LogisticRegression 학습 중...
  [VIF10] XGBoost 학습 중...
  [VIF10] LightGBM 학습 중...
  [VIF10_ALPHA005] LogisticRegression 학습 중...
  [VIF10_ALPHA005] XGBoost 학습 중...
  [VIF10_ALPHA005] LightGBM 학습 중...
  [VIF5_AIC] LogisticRegression 학습 중...
  [VIF5_AIC] XGBoost 학습 중...
  [VIF5_AIC] LightGBM 학습 중...


SMOTE sampling_strategy = 0.3
  After SMOTE → 0: 31087, 1: 9326
  [VIF5] LogisticRegression 학습 중...
  [VIF5] XGBoost 학습 중...
  [VIF5] LightGBM 학습 중...
  [VIF10] LogisticRegression 학습 중...
  [VIF10] XGBoost 학습 중...
  [VIF10] LightGBM 학습 중...
  [VIF10_ALPHA005] LogisticRegression 학습 중...
  [VIF10_ALPHA005] XGBoost 학습 중...
  [VIF10_ALPHA005] LightGBM 학습 중...
  [VIF5_AIC] LogisticRegression 학습 중...
  [VIF5_AIC] XGBoost 학습 중...
  [VIF5_AIC] LightGBM 학습 중...


SMOTE sampling_strategy = 0.5
  After SMOTE → 0: 31087, 1: 15543
 

SMOTE 후 데이터가 많다면 먼저 ratio=0.3 한 가지 조건으로 SVM 단독 테스트해보고 속도 확인한 다음 전체 돌리는 걸 추천해.

### LSTM
train / test DataFrame에 사업자등록번호, 회계년도 컬럼이 있어야 시퀀스 생성 가능

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import pandas as pd

from imblearn.over_sampling import BorderlineSMOTE
from ctgan import CTGAN

from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    precision_score, recall_score, f1_score
)


# ============================================================
# LSTM 모델 정의
# ============================================================

class LSTMClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :]).squeeze()


# ============================================================
# 시계열 시퀀스 생성 함수
# ============================================================

def make_sequences(df, feature_cols, target_col,
                   time_col="회계년도", id_col="사업자등록번호", window=3):
    X_list, y_list = [], []
    for _, group in df.groupby(id_col):
        group = group.sort_values(time_col).reset_index(drop=True)
        X = group[feature_cols].values
        y = group[target_col].values
        for i in range(len(group) - window + 1):
            X_list.append(X[i : i + window])
            y_list.append(y[i + window - 1])
    return np.array(X_list, dtype=np.float32), np.array(y_list, dtype=np.float32)


# ============================================================
# 오버샘플링 함수 정의
# ============================================================

def oversample_none(X_seq_train, y_seq_train, ratio=None):
    """클래스 불균형 처리 없음 - 원본 시퀀스 그대로 반환"""
    return X_seq_train, y_seq_train


def oversample_borderline_smote(X_seq_train, y_seq_train, ratio):
    """Flatten → BorderlineSMOTE → Reshape"""
    n_samples  = X_seq_train.shape[0]
    window     = X_seq_train.shape[1]
    n_features = X_seq_train.shape[2]

    X_flat = X_seq_train.reshape(n_samples, window * n_features)

    smote = BorderlineSMOTE(
        sampling_strategy=ratio,
        random_state=RANDOM_STATE,
        kind="borderline-1"
    )
    X_flat_res, y_res = smote.fit_resample(X_flat, y_seq_train)
    X_seq_res = X_flat_res.reshape(-1, window, n_features)

    return X_seq_res, y_res


def oversample_ctgan(X_seq_train, y_seq_train, ratio):
    """마지막 시점 단면 데이터로 CTGAN 학습 → 합성 → 시퀀스 복원"""
    window     = X_seq_train.shape[1]
    n_features = X_seq_train.shape[2]

    n_majority = int((y_seq_train == 0).sum())
    n_minority = int((y_seq_train == 1).sum())
    n_target   = int(n_majority * ratio)
    n_to_gen   = max(n_target - n_minority, 0)

    if n_to_gen == 0:
        return X_seq_train, y_seq_train

    minority_last = X_seq_train[y_seq_train == 1, -1, :]
    minority_df   = pd.DataFrame(minority_last)

    ctgan = CTGAN(epochs=100, verbose=False)
    ctgan.fit(minority_df, discrete_columns=[])
    synthetic_last = ctgan.sample(n_to_gen).values.astype(np.float32)

    minority_seqs = X_seq_train[y_seq_train == 1]
    mean_prefix   = minority_seqs.mean(axis=0)

    synthetic_seqs = np.tile(mean_prefix[np.newaxis, :, :], (n_to_gen, 1, 1))
    synthetic_seqs[:, -1, :] = synthetic_last

    X_seq_res = np.concatenate([X_seq_train, synthetic_seqs], axis=0)
    y_res      = np.concatenate([
        y_seq_train,
        np.ones(n_to_gen, dtype=np.float32)
    ])

    return X_seq_res, y_res


# ============================================================
# LSTM 학습 및 평가 공통 함수
# ============================================================

def train_and_evaluate_lstm(X_seq_res, y_res, X_seq_test, y_seq_test,
                             n_features, epochs=30, batch_size=64, lr=1e-3):

    X_tr = torch.tensor(X_seq_res,  dtype=torch.float32)
    y_tr = torch.tensor(y_res,       dtype=torch.float32)
    X_te = torch.tensor(X_seq_test,  dtype=torch.float32)

    loader = DataLoader(
        TensorDataset(X_tr, y_tr),
        batch_size=batch_size,
        shuffle=True
    )

    model     = LSTMClassifier(input_size=n_features)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    # None 방식은 불균형 그대로 → pos_weight로 보완
    n_neg      = (y_res == 0).sum()
    n_pos      = (y_res == 1).sum()
    pos_weight = torch.tensor([n_neg / n_pos])
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    model.train()
    for _ in range(epochs):
        for xb, yb in loader:
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        y_prob = torch.sigmoid(model(X_te)).numpy()

    y_pred = (y_prob >= THRESHOLD).astype(int)

    return {
        "Accuracy"  : accuracy_score(y_seq_test, y_pred),
        "ROC_AUC"   : roc_auc_score(y_seq_test, y_prob),
        "PR_AUC"    : average_precision_score(y_seq_test, y_prob),
        "F1"        : f1_score(y_seq_test, y_pred),
        "Precision" : precision_score(y_seq_test, y_pred),
        "Recall"    : recall_score(y_seq_test, y_pred)
    }


# ============================================================
# LSTM 실험 루프
# 방식별 ratio 설정이 다르므로 method_configs로 관리
# 총 조합 : None(1) + SMOTE(3) + CTGAN(3) × 4 feature × 3 window = 84개
# ============================================================

WINDOW_SIZES = [2, 3, 5]

method_configs = {
    # 방식명             : (함수,                      ratio 리스트)
    "None"              : (oversample_none,            [None]),
    "BorderlineSMOTE"   : (oversample_borderline_smote,[0.2, 0.3, 0.5]),
    "CTGAN"             : (oversample_ctgan,           [0.2, 0.3, 0.5])
}

lstm_results = []

for method_name, (oversample_fn, ratios) in method_configs.items():

    print("\n")
    print("="*70)
    print(f"[LSTM] 오버샘플링 방식 : {method_name}")
    print("="*70)

    for feature_name, feature_path in feature_files.items():

        feature_df   = pd.read_parquet(feature_path)
        feature_col  = (
            "final_feature" if "final_feature" in feature_df.columns else "feature"
        )
        use_features = [f for f in feature_df[feature_col].tolist() if f in train.columns]

        print(f"\n  Feature Set : {feature_name} ({len(use_features)}개)")

        for window in WINDOW_SIZES:

            X_seq_train, y_seq_train = make_sequences(
                train, use_features, TARGET_COL, window=window
            )
            X_seq_test, y_seq_test = make_sequences(
                test, use_features, TARGET_COL, window=window
            )

            if (y_seq_train == 1).sum() == 0 or (y_seq_test == 1).sum() == 0:
                print(f"  window={window} → 부실 샘플 없음, 스킵")
                continue

            for ratio in ratios:

                print(f"  window={window}, ratio={ratio} 오버샘플링 중...")

                X_seq_res, y_res = oversample_fn(X_seq_train, y_seq_train, ratio)

                print(f"    → 0: {int((y_res==0).sum())}, 1: {int((y_res==1).sum())}")

                metrics = train_and_evaluate_lstm(
                    X_seq_res, y_res,
                    X_seq_test, y_seq_test,
                    n_features=len(use_features)
                )

                lstm_results.append({
                    "Method"      : method_name,
                    "Feature_Set" : feature_name,
                    "Window"      : window,
                    "SMOTE_Ratio" : ratio if ratio is not None else "-",
                    "Num_Features": len(use_features),
                    **metrics
                })


# ============================================================
# 결과 출력 및 저장
# ============================================================

lstm_df = pd.DataFrame(lstm_results).round(4)

print("\n")
print("="*100)
print("LSTM 오버샘플링 방식 비교 결과")
print("="*100)
print(lstm_df.to_string(index=False))

# 방식별 평균 성능 비교 ← 핵심
print("\n")
print("="*70)
print("방식별 평균 성능 비교")
print("="*70)
print(
    lstm_df.groupby("Method")[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)

# 방식 × window별 평균
print("\n")
print("="*70)
print("방식 × Window별 평균 성능")
print("="*70)
print(
    lstm_df.groupby(["Method", "Window"])[["ROC_AUC", "PR_AUC", "F1", "Recall"]]
    .mean()
    .round(4)
)

lstm_df.to_csv(
    "LSTM_Oversample_Comparison.csv",
    index=False,
    encoding="utf-8-sig"
)

print("\n저장 완료 : LSTM_Oversample_Comparison.csv")